# Tycho Survey `v3` Stage 2 — Pre-Registered 5-Arm Causal Surgery (Colab G4 / RTX PRO 6000)

**Spec:** `docs/v3_stage2_spec.md` on branch `v3-roadmap` (`memess2001/tycho-survey`).

**Recommended Runtime:** Colab **G4** (`NVIDIA RTX PRO 6000 Blackwell`, 96 GB VRAM, 48 vCPU, ~177 GB RAM, ~8.8 CU/h).
- **Cell 1–2:** Mount Google Drive & clone/pull `v3-roadmap`.
- **Cell 3:** Pack `1.05B` tokens of `FineWeb-Edu` into local NVMe `train.bin` (`2.1 GB`) + `val.bin` (`4 MB`) in ~2–4 min, and cache once in Google Drive.
- **Cell 4 (Pilot Seed 42, ~2.5–3h total):** Run all 5 arms (`post_olmo2`, `post_matched_gamma`, `post_matched_emb`, `post_frozen_gamma`, `pre_ln`) with `--compile`, `--resume`, and live Google Drive mirroring.
- **Cell 5 (Confirmation Seeds 137 & 2026, ~5–6h total):** Run remaining 10 runs after verifying Seed 42 scorecard.

In [ ]:
# Cell 1: Mount Google Drive & verify G4 hardware specs
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!echo "CPUs: $(nproc)" && free -h && df -h /content

In [ ]:
# Cell 2: Clone/pull tycho-survey (branch v3-roadmap) & install lightweight deps
import os
REPO_DIR = '/content/tycho-survey'
if not os.path.exists(REPO_DIR):
    !git clone -b v3-roadmap https://github.com/memess2001/tycho-survey.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout v3-roadmap && git pull origin v3-roadmap

!pip -q install -U datasets transformers
!cd {REPO_DIR} && git log -n 3 --oneline

In [ ]:
# Cell 3: Prepare FineWeb-Edu 1.05B-token uint16 binary shard (cached in Google Drive)
import os, shutil
from pathlib import Path

LOCAL_DATA = Path('/content/stage2_data')
DRIVE_ROOT = Path('/content/drive/MyDrive/tycho_v3_stage2')
DRIVE_DATA = DRIVE_ROOT / 'data'
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

# If already packed in Google Drive, copy fast to local NVMe SSD (~15s)
if (DRIVE_DATA / 'train.bin').exists() and (DRIVE_DATA / 'val.bin').exists() and (DRIVE_DATA / 'dataset_meta.json').exists():
    for fn in ['train.bin', 'val.bin', 'dataset_meta.json']:
        if not (LOCAL_DATA / fn).exists():
            print(f'Copying {fn} from Drive -> local SSD...')
            shutil.copy2(DRIVE_DATA / fn, LOCAL_DATA / fn)
else:
    print('Packing 1.05B FineWeb-Edu tokens on local SSD (~2-4 min on 48 vCPUs)...')
    !python3 /content/tycho-survey/scripts/v3/prep_fineweb_bin.py \
        --out-dir /content/stage2_data \
        --train-tokens 1050000000 \
        --val-tokens 2000000 \
        --batch-docs 512
    for fn in ['train.bin', 'val.bin', 'dataset_meta.json']:
        print(f'Backing up {fn} -> Google Drive...')
        shutil.copy2(LOCAL_DATA / fn, DRIVE_DATA / fn)

!ls -lh /content/stage2_data && cat /content/stage2_data/dataset_meta.json

In [ ]:
# Cell 4: Pilot Seed 42 — Run ALL 5 pre-registered arms (~2.5-3h total on G4 PRO 6000)
# Safe to re-run after any Colab disconnect: auto-resumes from latest.pt & skips DONE arms!
!python3 /content/tycho-survey/scripts/v3/run_stage2_suite.py \
    --seeds 42 \
    --preset stage2_70m \
    --dataset bin \
    --train-bin /content/stage2_data/train.bin \
    --val-bin /content/stage2_data/val.bin \
    --out-root /content/stage2_runs \
    --sync-root /content/drive/MyDrive/tycho_v3_stage2/runs \
    --sync-latest \
    --save-latest-every 1000 \
    --compile

In [ ]:
# Cell 5: Confirmation Seeds 137 & 2026 (run after inspecting Seed 42 scorecard above!)
!python3 /content/tycho-survey/scripts/v3/run_stage2_suite.py \
    --seeds 42 137 2026 \
    --preset stage2_70m \
    --dataset bin \
    --train-bin /content/stage2_data/train.bin \
    --val-bin /content/stage2_data/val.bin \
    --out-root /content/stage2_runs \
    --sync-root /content/drive/MyDrive/tycho_v3_stage2/runs \
    --sync-latest \
    --save-latest-every 1000 \
    --compile